In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
import os 

## import helper 

In [2]:
platformID = 'FBE'

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from functions import lookup_loader, execute_sql_query
import test_functions

from config import gam_info

In [4]:
lookup = lookup_loader(gam_info, platformID, '1e',
                       with_country=False)
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']


✅ Test FBE_1e_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1e_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1e_02 passed: No empty values in lookup.
...updating logbook...

✅ Test FBE_1e_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1e_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1e_05 passed: No empty values in lookup.
...updating logbook...



# engagements 

In [5]:
sql_query = f"""
    SELECT
        week_commencing,
        page_id,
        CASE
            WHEN (AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user) 
                 > AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer)
            THEN ((AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user)) 
                 + (AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer))*0.04827
            ELSE (AVG(video_views)/AVG(page_video_views_to_10s_unique_viewer)) 
                 + ((AVG(engagements)/AVG(post_engagements_to_page_consumptions))/AVG(avg_engagements_per_user))*0.04822
        END AS engaged_reach
    FROM 
        redshiftdb.central_insights.adverity_social_facebook_by_page AS p
    RIGHT JOIN
        world_service_audiences_insights.social_media_lookup_fb AS l
        ON p.page_id = l.fb_page_id
    WHERE 
        period = 'week'
    AND
        week_commencing BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['w/c_end']}'
    GROUP BY 
        week_commencing, page_id
        ;
"""

file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_engagements_redshift_extract.csv"

# TODO make this more fail safe - e.g. I want to run this and say if it should query redshift or not
try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+df['page_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")
    
facebook_engagements_raw = pd.read_csv(file, keep_default_na=False)
facebook_engagements_raw['page_id'] = facebook_engagements_raw['page_id'].astype(str)
facebook_engagements_raw['week_commencing'] = pd.to_datetime(facebook_engagements_raw['week_commencing'])
facebook_engagements_raw = facebook_engagements_raw.rename(columns={'page_id': 'Channel ID', 
                                                                    'week_commencing': 'w/c'})
print(facebook_engagements_raw.shape)


no redshift connection using last time queried
(4259, 3)


In [6]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_engagements_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1e_06",
                                             "Check that all page IDs are found in SQL")

# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=facebook_engagements_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1e_07",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_engagements_raw, 
                           numeric_columns=['engaged_reach'], 
                           test_number=f"{platformID}_1e_08",
                           test_step='Check no missing values in engaged_reach column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_engagements_raw, 
                               ['Channel ID', 'w/c'], 
                               test_number=f"{platformID}_1e_09",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test FBE_1e_06 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
              Channel ID      Start End        w/c
1725  FBE171824429536304 2025-04-02 NaT 2025-04-28
1727  FBE171824429536304 2025-04-02 NaT 2025-05-12
1729  FBE171824429536304 2025-04-02 NaT 2025-05-26
1742  FBE171824429536304 2025-04-02 NaT 2025-08-25
1743  FBE171824429536304 2025-04-02 NaT 2025-09-01
1744  FBE171824429536304 2025-04-02 NaT 2025-09-08
1745  FBE171824429536304 2025-04-02 NaT 2025-09-15
1746  FBE171824429536304 2025-04-02 NaT 2025-09-22
1747  FBE171824429536304 2025-04-02 NaT 2025-09-29
1748  FBE171824429536304 2025-04-02 NaT 2025-10-06
1749  FBE171824429536304 2025-04-02 NaT 2025-10-13
1750  FBE171824429536304 2025-04-02 NaT 2025-10-20
1751  FBE171824429536304 2025-04-02 NaT 2025-10-27
1752  FBE171824429536304 2025-04-02 NaT 2025-11-03
1753  FBE171824429536304 2025-04-02 NaT 2025-11-10
1754  FBE171824429536304 2025-04-02 NaT 2025-11-17
1755  

## processing engagements

In [7]:
# needs to be right because we want to make sure that all the accounts in lookup are there 
facebook_engagements = facebook_engagements_raw.merge(socialmedia_accounts[['Channel ID', 'ServiceID', 'Start', 'End']], 
                                                      on='Channel ID', how='right', indicator=True)
test_functions.test_inner_join(facebook_engagements_raw, socialmedia_accounts, 
                               ['Channel ID'], 
                               f"{platformID}_1e_10", 
                               test_step='checking social media accounts in lookup, adding service',
                               focus='right')


mask = (
    (facebook_engagements['w/c'] >= facebook_engagements['Start']) &
    (
        facebook_engagements['End'].isna() |
        (facebook_engagements['w/c'] <= facebook_engagements['End'])
    )
)
# there are newly active accounts that have 0 for the weeks before they became active
# for accurate reach calculation those weeks need to be removed
facebook_engagements = facebook_engagements[mask]


✅ Inner join test FBE_1e_10 successful: No issues found.
...updating logbook...



In [8]:

facebook_engagements[facebook_engagements['Channel ID'] == 'FBE102775241582303'].sort_values('w/c')

,w/c,Channel ID,engaged_reach,ServiceID,Start,End,_merge
214,2025-03-31,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
229,2025-04-07,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
240,2025-04-14,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
246,2025-04-21,FBE102775241582303,0.5316,WOR,2020-04-01,NaT,both
218,2025-04-28,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
212,2025-05-05,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
224,2025-05-12,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
239,2025-05-19,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
215,2025-05-26,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both
250,2025-06-02,FBE102775241582303,0.0000,WOR,2020-04-01,NaT,both


In [9]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'ServiceID', 'w/c', 'engaged_reach']
facebook_engagements[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",
                                       index=None)
